# 03. 윈도잉 기반 피처 엔지니어링

진동 신호와 공정 파라미터에서 모델 학습용 피처 테이블을 만든다.

- **윈도잉** — 2048 샘플(≈170ms @ 12kHz), 50% 오버랩
- **시간 도메인** — RMS, Peak, Crest/Impulse/Shape/Margin Factor, Skewness, Kurtosis, Zero-crossing, Energy
- **주파수 도메인** — FFT / Band power / Welch PSD / 스펙트럴 엔트로피·중심
- **베어링 특성 주파수(BPFI/BPFO/BSF/FTF)** — Hilbert 포락선 스펙트럼에서 추출 (결함 진단 표준)

대응 모듈: `src/career_kia/features/{windowing,time_domain,freq_domain}.py`

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from career_kia.config import CWRU_FS_12K, INTERIM_DIR, PROCESSED_DIR, WINDOW_SIZE, WINDOW_STRIDE
from career_kia.features import freq_domain, time_domain
from career_kia.features.windowing import make_windows

plt.rcParams['figure.figsize'] = (11, 3)
plt.rcParams['axes.grid'] = True

## 1. 윈도잉 동작 확인

In [ ]:
# 전처리된 배치 샘플
batch = pd.read_parquet(INTERIM_DIR / 'preprocessed_batch.parquet')
ir_row = batch[batch['vibration_fault_type'] == 'IR'].iloc[0]
sig = np.load(INTERIM_DIR / 'cwru_clean' / ir_row['vibration_file'])
print(f'신호 길이: {len(sig)} 샘플, fs = 12 kHz')

W = make_windows(sig, window_size=WINDOW_SIZE, stride=WINDOW_STRIDE)
print(f'윈도우: {W.shape} — ({WINDOW_SIZE} 샘플, stride {WINDOW_STRIDE})')

## 2. 시간 도메인 피처

In [ ]:
tfeat = time_domain.compute_time_features(W)
print('시간 도메인 피처 (IR 결함 신호의 윈도우별):')
df_t = pd.DataFrame(tfeat)
print(df_t.describe().round(4))

## 3. 주파수 도메인 — 포락선 스펙트럼

In [ ]:
# 3개 결함 유형 신호의 포락선 스펙트럼 비교
fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
for ax, cls in zip(axes, ['Normal', 'IR', 'OR', 'Ball']):
    row = batch[batch['vibration_fault_type'] == cls].iloc[0]
    s = np.load(INTERIM_DIR / 'cwru_clean' / row['vibration_file'])
    freqs, env = freq_domain.envelope_spectrum(s.reshape(1, -1), fs=CWRU_FS_12K)
    ax.plot(freqs, env[0], lw=0.7)
    for name, f0 in freq_domain.DEFAULT_FAULT_FREQS_HZ.items():
        ax.axvline(f0, color='r', lw=0.5, alpha=0.4)
        ax.text(f0, env[0].max() * 0.9, name, rotation=90, fontsize=7, va='top')
    ax.set_title(f'{cls} — 포락선 스펙트럼')
    ax.set_xlim(0, 300)
axes[-1].set_xlabel('주파수 (Hz)')
plt.tight_layout()
plt.show()

## 4. 배치 단위 피처 집계

`make features` 로 실행되는 `run_pipeline.py` 는 배치별로 윈도우 피처를 생성하고 mean/max/std로 집계한다. 아래는 그 결과의 도메인 적합성(sanity) 점검.

In [ ]:
df = pd.read_parquet(PROCESSED_DIR / 'features.parquet')
print(f'피처 테이블: {df.shape}')

print('\n결함 유형별 시간 도메인 평균:')
print(df.groupby('vibration_fault_type')[['t_rms_mean', 't_kurtosis_max', 't_crest_factor_max']].mean().round(3))

print('\n결함 유형별 베어링 특성 주파수 포락선 진폭 (max):')
print(df.groupby('vibration_fault_type')[['f_env_BPFI_max', 'f_env_BPFO_max', 'f_env_BSF_max', 'f_env_FTF_max']].mean().round(4))

In [ ]:
# 주요 피처 분포 비교 (결함 유형별)
import seaborn as sns
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, col in zip(axes, ['t_kurtosis_max', 't_crest_factor_max', 'f_env_BPFO_max']):
    sns.boxplot(data=df, x='vibration_fault_type', y=col, ax=ax, order=['Normal','IR','OR','Ball'])
    ax.set_title(col)
plt.tight_layout()
plt.show()

## 요약

- **윈도우 수**: 배치당 약 10개 (1초 신호 / 2048 samples / stride 1024)
- **시간 피처 14종 × 3 집계 = 42개**
- **주파수 피처**: 스펙트럴 엔트로피/중심 + 6 대역 power + 4 특성 주파수 × 2 집계 = 24개
- **공정 파라미터 5 + 메타 6 + 라벨 7 + 신호 피처 66 = 총 84 컬럼**

결함 유형별 sanity check 결과:
- Outer Race(OR) 결함 → BPFO 포락선 진폭 최대
- Ball 결함 → BSF 포락선 진폭 최대
- 결함 신호 전반의 Kurtosis/Crest Factor가 Normal보다 높음

→ **피처가 물리적 결함 메커니즘을 반영**하고 있어 Phase 5 모델 학습에 유의미한 신호를 제공한다.